# Fabric Monitoring — Data Prep

Persists billing, chargeback, and capacity metrics data into Delta tables in this Lakehouse.
Run daily (via schedule or pipeline) to accumulate history beyond the apps' retention windows.

**Tables created:**

| Table | Source | Refresh Mode | Rows |
|-------|--------|-------------|------|
| `billing_fabric` | Azure Cost Export parquets (shortcut) | Overwrite | ~1,300 |
| `chargeback_facts` | Chargeback App (DAX API) | Append + dedup | Grows daily |
| `chargeback_domains` | Chargeback App (DAX API) | Overwrite | Small |
| `capacity_capacities` | Capacity Metrics App (DAX API) | Overwrite | Small |
| `capacity_workspaces` | Capacity Metrics App (DAX API) | Overwrite | Small |

**Prerequisites:** Attach this notebook to the `MonitoringLakehouse` in the MonitoringAdmin workspace.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

CM_WORKSPACE_ID = "87d74804-68a9-4e43-8742-4d295cecd2ea"
CM_DATASET_ID = "1d1a8613-f490-4ba8-9bb4-72324c3a747e"

CB_WORKSPACE_ID = "6741112c-52c2-42e4-8e6f-5e5159d28111"
CB_DATASET_ID = "1df83a07-9420-4ac3-9e7c-90853fb6d238"

COST_EXPORT_FOLDER = "fabric_amortized_cost"

PBI_API_BASE = "https://api.powerbi.com/v1.0/myorg"

In [ ]:
import glob, os, time
import pandas as pd
import requests
from pyspark.sql.functions import col, lit, current_timestamp

def get_token():
    try:
        from notebookutils import mssparkutils
        return mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
    except ImportError:
        from azure.identity import DefaultAzureCredential
        return DefaultAzureCredential().get_token("https://analysis.windows.net/powerbi/api/.default").token

def execute_dax(ws_id, ds_id, dax, retries=3):
    url = f"{PBI_API_BASE}/groups/{ws_id}/datasets/{ds_id}/executeQueries"
    payload = {"queries": [{"query": dax}], "serializerSettings": {"includeNulls": True}}
    for attempt in range(retries):
        headers = {"Authorization": f"Bearer {get_token()}", "Content-Type": "application/json"}
        resp = requests.post(url, headers=headers, json=payload)
        if resp.status_code == 200:
            rows = resp.json()["results"][0]["tables"][0]["rows"]
            if not rows: return pd.DataFrame()
            df = pd.DataFrame(rows)
            df.columns = [c.strip("[]").split("[")[-1].rstrip("]") for c in df.columns]
            return df
        if resp.status_code == 429:
            time.sleep(int(resp.headers.get("Retry-After", 30)))
            continue
        if resp.status_code == 401 and attempt < retries - 1:
            continue
        resp.raise_for_status()
    raise RuntimeError("Failed")

def query_cm(dax): return execute_dax(CM_WORKSPACE_ID, CM_DATASET_ID, dax)
def query_cb(dax): return execute_dax(CB_WORKSPACE_ID, CB_DATASET_ID, dax)

print("Ready")

---
## 1. Billing → `billing_fabric`

Read cost export parquets from shortcut, filter to `Microsoft.Fabric`, write as Delta table. **Overwrite** each run (source parquets are the system of record).

In [ ]:
# ============================================================================
# billing_fabric — Azure billing filtered to Fabric capacities only
# Mode: OVERWRITE (parquets in shortcut are the source of truth)
# ============================================================================

COST_EXPORT_PATH = f"/lakehouse/default/Files/{COST_EXPORT_FOLDER}"

if os.path.exists(COST_EXPORT_PATH):
    parquet_files = sorted(glob.glob(f"{COST_EXPORT_PATH}/*/*/*.parquet"))
    print(f"Found {len(parquet_files)} parquet files")

    if parquet_files:
        dfs = []
        for f in parquet_files:
            parts = f.replace(COST_EXPORT_PATH + "/", "").split("/")
            df_part = pd.read_parquet(f)
            df_part["_export_period"] = parts[0]
            dfs.append(df_part)

        df_raw = pd.concat(dfs, ignore_index=True)
        df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
        df_raw["costInBillingCurrency"] = pd.to_numeric(df_raw["costInBillingCurrency"], errors="coerce").fillna(0)

        # Filter to Fabric only
        df_billing = df_raw[df_raw["consumedService"] == "Microsoft.Fabric"].copy()

        # Extract capacity name from ResourceId
        df_billing["capacity_name"] = (
            df_billing["ResourceId"].astype(str)
            .str.rsplit("/", n=1).str[-1]
            .str.lower().str.strip()
        )

        # Deduplicate overlapping exports
        df_billing = df_billing.drop_duplicates(
            subset=["date", "capacity_name", "meterName", "chargeType", "costInBillingCurrency"]
        )

        # Write to Delta
        spark.createDataFrame(df_billing).write.mode("overwrite").format("delta").saveAsTable("billing_fabric")
        count = spark.sql("SELECT COUNT(*) as c FROM billing_fabric").collect()[0]["c"]
        print(f"billing_fabric: {count:,} rows written")
    else:
        print("No parquet files found.")
else:
    print(f"Path not found: {COST_EXPORT_PATH}")
    print("Attach this notebook to the Lakehouse with the cost export shortcut.")

---
## 2. Capacity Metrics → `capacity_capacities`, `capacity_workspaces`

Dimensions from the Capacity Metrics App. **Overwrite** — always current.

In [ ]:
# ============================================================================
# capacity_capacities — Capacity inventory
# capacity_workspaces — Workspace-to-capacity mapping
# Mode: OVERWRITE
# ============================================================================

df_caps = query_cm("""
    EVALUATE SELECTCOLUMNS('Capacities',
        "Capacity Id", 'Capacities'[Capacity Id],
        "Capacity Name", 'Capacities'[Capacity name],
        "SKU", 'Capacities'[SKU],
        "Region", 'Capacities'[Region],
        "State", 'Capacities'[State]
    )
""")

# Add core count from Chargeback
cb_caps = query_cb('EVALUATE SELECTCOLUMNS(\'Capacities\', "Capacity Id", \'Capacities\'[Capacity Id], "Core count", \'Capacities\'[Core count])')
df_caps = df_caps.merge(cb_caps, on="Capacity Id", how="left")
df_caps["Capacity Name Lower"] = df_caps["Capacity Name"].str.lower()

spark.createDataFrame(df_caps).write.mode("overwrite").format("delta").saveAsTable("capacity_capacities")
print(f"capacity_capacities: {len(df_caps)} rows")

# Workspaces
df_ws = query_cm("""
    EVALUATE SELECTCOLUMNS('Workspaces',
        "Workspace Id", 'Workspaces'[Workspace Id],
        "Workspace Name", 'Workspaces'[Workspace name],
        "Capacity Id", 'Workspaces'[Capacity Id]
    )
""")

spark.createDataFrame(df_ws).write.mode("overwrite").format("delta").saveAsTable("capacity_workspaces")
print(f"capacity_workspaces: {len(df_ws)} rows")

---
## 3. Chargeback → `chargeback_facts`, `chargeback_domains`

The Chargeback App only retains 14-30 days. We **append** new rows and **deduplicate** so that over time the Delta table accumulates months of history that the app itself can't provide.

In [ ]:
# ============================================================================
# chargeback_domains — Domain/subdomain mapping
# Mode: OVERWRITE (small dimension)
# ============================================================================

df_domains = query_cb("""
    EVALUATE SELECTCOLUMNS('Domains',
        "Domain unique key", 'Domains'[Domain unique key],
        "Domain", 'Domains'[Domain],
        "Subdomain", 'Domains'[Subdomain]
    )
""")

spark.createDataFrame(df_domains).write.mode("overwrite").format("delta").saveAsTable("chargeback_domains")
print(f"chargeback_domains: {len(df_domains)} rows")

In [ ]:
# ============================================================================
# chargeback_facts — Daily CU per workspace/user/item/operation
# Mode: APPEND + DEDUPLICATE
#
# Each run pulls the current 14-day window from the Chargeback App.
# Appends to the Delta table, then removes duplicate rows.
# Over time, this builds months of chargeback history.
# ============================================================================

df_cb = query_cb("""
    EVALUATE SELECTCOLUMNS('Chargeback',
        "Capacity Id", 'Chargeback'[Capacity Id],
        "Date", 'Chargeback'[Date],
        "Item Id", 'Chargeback'[Item Id],
        "Workspace Id", 'Chargeback'[Workspace Id],
        "Operation name", 'Chargeback'[Operation name],
        "User", 'Chargeback'[User],
        "CU (s)", 'Chargeback'[CU (s)],
        "Duration (s)", 'Chargeback'[Duration (s)],
        "Operations", 'Chargeback'[Operations],
        "Experience", 'Chargeback'[Experience],
        "Domain unique key", 'Chargeback'[Domain unique key]
    )
""")

df_cb["Date"] = pd.to_datetime(df_cb["Date"])
df_cb["_ingested_at"] = pd.Timestamp.now()

spark_cb = spark.createDataFrame(df_cb)
date_min = df_cb["Date"].min().date()
date_max = df_cb["Date"].max().date()
print(f"Chargeback pull: {len(df_cb):,} rows | {date_min} to {date_max}")

# Check if table exists
table_exists = spark.catalog.tableExists("chargeback_facts")

if not table_exists:
    # First run: create the table
    spark_cb.write.format("delta").saveAsTable("chargeback_facts")
    print(f"chargeback_facts: created with {len(df_cb):,} rows")
else:
    # Subsequent runs: merge (upsert) to avoid duplicates
    # Use Delta MERGE — match on the natural key, update if exists, insert if new
    from delta.tables import DeltaTable

    target = DeltaTable.forName(spark, "chargeback_facts")

    merge_condition = """
        target.`Capacity Id` = source.`Capacity Id`
        AND target.Date = source.Date
        AND target.`Workspace Id` = source.`Workspace Id`
        AND target.`Item Id` = source.`Item Id`
        AND target.`Operation name` = source.`Operation name`
        AND target.User = source.User
        AND target.Experience = source.Experience
    """

    (
        target.alias("target")
        .merge(spark_cb.alias("source"), merge_condition)
        .whenMatchedUpdateAll()  # Update if values changed
        .whenNotMatchedInsertAll()  # Insert new rows
        .execute()
    )

    total = spark.sql("SELECT COUNT(*) as c FROM chargeback_facts").collect()[0]["c"]
    print(f"chargeback_facts: merged. Total rows now: {total:,}")

# Show date coverage
coverage = spark.sql("""
    SELECT MIN(Date) as earliest, MAX(Date) as latest, COUNT(DISTINCT Date) as days
    FROM chargeback_facts
""").collect()[0]
print(f"Date coverage: {coverage['earliest']} to {coverage['latest']} ({coverage['days']} days)")

---
## 4. Summary

Show all tables in the Lakehouse and their row counts.

In [ ]:
# ============================================================================
# SUMMARY — All tables in this Lakehouse
# ============================================================================

tables = spark.sql("SHOW TABLES").collect()
print(f"{'Table':<30} {'Rows':>10}")
print("-" * 42)
for t in tables:
    name = t["tableName"]
    try:
        count = spark.sql(f"SELECT COUNT(*) as c FROM {name}").collect()[0]["c"]
        print(f"{name:<30} {count:>10,}")
    except:
        print(f"{name:<30} {'error':>10}")